In [ ]:
# Databricks notebook source
# tutorial_name: 03-Day9-Extras-Course-Review-and-Extensions
# notebookName: 03-Day9-Extras-Course-Review-and-Extensions
# COMMAND ----------

# DBTITLE 1,Paths
notepoint_rel = "hands-on/day-09/notebooks/03-Day9-Extras-Course-Review-and-Extensions.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
DAY6_ROOT = f"{BASE_PATH}day06-{STUDENT_ID}"
DAY7_ROOT = f"{BASE_PATH}day07-{STUDENT_ID}"
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
DAY9_ROOT = f"{BASE_PATH}day09-{STUDENT_ID}"
# COMMAND ----------

# DBTITLE 1,Prerequisite
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: P_BASIC")
except Exception as e:
    print("P_BASIC:", e)
# COMMAND ----------


# Day 9 supplement — course review and extensions

Use this notebook when the schedule allows. It is longer than the other Day 9 labs and revisits earlier topics with figures, short code cells, and a few open prompts.

Run top to bottom, or jump to the section for a given day from the notebook outline.

Treat it as review unless your instructor assigns specific sections.


## Figure — entire course arc

```mermaid
flowchart LR
  D1[D1 Foundations] --> D2[D2 DataFrames]
  D2 --> D3[D3 Medallion]
  D3 --> D4[D4 Unity Catalog]
  D4 --> D5[D5 Delta deep]
  D5 --> D6[D6 Ops Delta]
  D6 --> D7[D7 Streaming DLT]
  D7 --> D8[D8 Jobs]
  D8 --> D9[D9 Monitor SQL]
```


## Day 1 — Foundations (recap)

**Themes:** SparkSession, lazy evaluation, SQL vs DataFrames, basic **widgets**.

**ASCII**

```text
  Driver (your notebook)  -->  Executors (cluster)
           lazy plan until action (.show, .count, .write)
```


In [ ]:
print("Spark version:", spark.version)


### Exercise — simple SQL in Spark
```python
spark.sql("SELECT 1 AS ok").show()
```


In [ ]:
spark.sql("SELECT 1 AS day1_check").show()


## Day 2 — DataFrames (recap)

**Themes:** `select`, `filter`, `groupBy`, **schemas**.


In [ ]:
spark.read.format("delta").load(P_BASIC).select("DEST_COUNTRY_NAME", "count").limit(5).show(truncate=False)


### Exercise — aggregation pattern

Same as early labs: **groupBy** + **sum**.


In [ ]:
from pyspark.sql import functions as F
spark.read.format("delta").load(P_BASIC).groupBy("DEST_COUNTRY_NAME").agg(F.sum("count").alias("s")).orderBy(F.desc("s")).limit(5).show(truncate=False)


## Day 3 — Medallion (recap)

**Bronze / Silver / Gold** — **Day 8** automated it with **Jobs**; **Day 9** monitors it.


In [ ]:
print("Day 8 gold path (if exists):", f"{DAY8_ROOT}/medallion/gold_by_destination")


## Day 4 — Unity Catalog (recap)

**Themes:** catalogs, schemas, **grants**, **lineage**, **audit** (`system.access.audit`).


In [ ]:
try:
    print("current_catalog:", spark.sql("SELECT current_catalog()").collect()[0][0])
except Exception as e:
    print("UC SQL:", e)


## Day 5 — Delta Lake (recap)

**ACID**, **history**, **time travel**, **MERGE** — production table format on lakehouse.


In [ ]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").show(truncate=False)


### Exercise — history size

More versions imply more **operational churn** to watch.


In [ ]:
n = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").count()
print("History rows:", n)


## Day 6 — Delta operations (recap)

**VACUUM**, **OPTIMIZE**, **ZORDER**, **time travel** — cost/latency tradeoffs.


In [ ]:
spark.sql(f"SELECT version, operation FROM (DESCRIBE HISTORY delta.`{P_BASIC}`) ORDER BY version DESC LIMIT 3").show(truncate=False)


## Day 7 — Streaming & DLT (recap)

**Structured Streaming**, **checkpoints**, **CDF**; **DLT** expectations and **pipelines**.


In [ ]:
print("Day 7 student prefix example:", DAY7_ROOT)


```mermaid
flowchart LR
  S[Stream] --> CP[Checkpoint]
  CP --> D[Delta sink]
  D --> CDF[Optional CDF read]
```


In [ ]:
# No local stream here — conceptual cell only
print("Review Day 7 notebook 01 for rate -> Delta + CDF.")


## Day 8 — Workflows (recap)

**Jobs**, **tasks**, **dependencies**, **parameters** / widgets.


In [ ]:
print("Day 8 paths:", DAY8_ROOT)


### Figure — dependency chain

```text
  bronze task --> silver task --> gold task
```


In [ ]:
print("If multi-task job configured, failures bubble to dependents.")


## Day 9 — Monitoring & SQL (recap)

**System tables**, **alerts**, **SQL warehouses**, **dashboards**.


In [ ]:
print("Use notebook 01 (monitoring) + 02 (SQL) for detail.")


## Cross-cutting — Security mindset

```mermaid
flowchart TB
  I[Identity] --> G[Grants UC]
  G --> D[Data ABFS]
  D --> L[Lineage Audit]
```


In [ ]:
print("Principle: least privilege on catalogs + storage ACLs.")


## Cross-cutting — Cost mindset

- Stop / auto-stop **warehouses**
- **Job** clusters vs always-on
- **Photon** where it pays off (platform dependent)


In [ ]:
print("Action: verify warehouse auto-stop in SQL workspace (manual).")


## Cross-cutting — Data quality quick checks

1. Null rates on keys  
2. Row counts vs prior day  
3. Schema **evolution** awareness


In [ ]:
df = spark.read.format("delta").load(P_BASIC)
from pyspark.sql import functions as F
df.select(F.sum(F.when(F.col("DEST_COUNTRY_NAME").isNull(), 1).otherwise(0)).alias("null_dest")).show()


## Extension — read CSV again (Day 5 source)

Reconnect **files** to **Delta** mindset.


In [ ]:
spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(3).show(truncate=False)


## Extension — MERGE mental model (no execution if unsure)

Use **Day 5** MERGE labs; pattern: **target** Delta, **source** batch, **match** keys.


In [ ]:
print("MERGE: WHEN MATCHED UPDATE ELSE INSERT — idempotent loads.")


## Extension — partition thinking

**Day 5** partitioned writes — **skip** pruning in production queries.


In [ ]:
print("WHERE on partition column avoids full scan.")


## Extension — streaming vs batch

**Batch:** overwrite / merge daily. **Stream:** micro-batches + checkpoint.


In [ ]:
print("Choose based on latency SLA + cost.")


## Extension — CI/CD one-liner

**Git** stores notebooks + job JSON; **pipeline** deploys to **staging** then **prod**.


In [ ]:
print("See databricks bundle / Terraform docs for your org standard.")


## Extension — certification pointers (self-study)

- Databricks **Data Engineer Associate** topics map to **Days 2–8**  
- **Administrator** adds identity networking **policies**


In [ ]:
print("Official: databricks.com/learn certifications")


## Extension — troubleshooting tree

```text
  Job failed?
    -> Task logs
    -> Cluster events
    -> Storage credentials
    -> Code change in last deploy?
```


In [ ]:
print("Keep correlation ids from Day 8 smoke / audit pattern.")


## Extension — documentation you should leave behind

- **Data dictionary** (gold tables)  
- **Runbook** for on-call (who restarts what)  
- **RPO/RTO** for pipelines


In [ ]:
print("Link dashboards + job names in one Confluence / Wiki page.")


## Extension — diversity of interfaces

Notebooks, **SQL**, **Jobs UI**, **REST**, **Terraform** — same platform.


In [ ]:
print("Pick interface by persona: engineer vs analyst vs platform.")


## Extension — Lakehouse vs legacy

| Legacy | Lakehouse |
|--------|-----------|
| Data warehouse only | Warehouse + ML + streaming on same store |
| Separate silos | UC governance across workloads |


In [ ]:
print("You trained on the lakehouse pattern end-to-end.")


## Final exercise — small completion marker under your Day 9 prefix

Skip this if you prefer not to write under **day09-{STUDENT_ID}**; it only checks that you can land a tiny Delta table there.


In [ ]:
spark.sql(f"SELECT '{STUDENT_ID}' AS student_id, current_timestamp() AS finished_at, 'course_review' AS note").write.format("delta").mode("overwrite").save(f"{DAY9_ROOT}/extras/course_completion_marker")
spark.read.format("delta").load(f"{DAY9_ROOT}/extras/course_completion_marker").show(truncate=False)


## Thank you / closeout

Re-run any **Day 1–8** notebook for weak areas; keep **`STUDENT_ID`** consistent across days when comparing paths.

**Extra materials folder:** `C:\\...\\databricks` — Spark SQL, monitoring markdowns, workflow labs.


In [ ]:
# dbutils.fs.rm(DAY9_ROOT, recurse=True)  # optional cleanup


## Reflection — streaming vs batch

If you have time: in one sentence, contrast batch processing with streaming for a pipeline you know.


In [ ]:
# Add a markdown cell with your sentence if you want it saved here.


## Reflection — governance

If you have time: note one place where Unity Catalog permissions matter in your workspace.


In [ ]:
# Add a markdown cell with your sentence if you want it saved here.


## Reflection — scheduling

If you have time: name one reason you would choose a scheduled job over an ad hoc notebook run.


In [ ]:
# Add a markdown cell with your sentence if you want it saved here.
